# Playbook 03 — Drug Signature Reversal

Take a disease signature (from playbook 02) and a library of drug signatures (LINCS or CMap), compute reversal scores, and rank compounds by how strongly they oppose the disease direction.

**Reversal score ∈ [-1, 1]:**
- +1 = drug perfectly reverses disease direction
-  0 = no overlap
- -1 = drug aggravates (mirrors disease direction)

In [ ]:
import logging, json
from pathlib import Path
logging.basicConfig(level=logging.INFO)
import numpy as np
import pandas as pd

## 1. Load disease signature

In [ ]:
# Synthetic disease: up = G0001-G0030, down = G0500-G0530
disease_up   = [f'GENE_{i:04d}' for i in range(30)]
disease_down = [f'GENE_{i:04d}' for i in range(500, 530)]

## 2. Build synthetic drug library (replace with LINCS in production)

In [ ]:
rng = np.random.default_rng(123)
drug_library = {}
# Drug A: perfect reversal
drug_library['drug_A'] = {'up_genes': disease_down, 'down_genes': disease_up}
# Drug B: partial reversal
drug_library['drug_B'] = {'up_genes': disease_down[:15] + [f'GENE_{i:04d}' for i in rng.integers(1000, 1500, 10)],
                          'down_genes': disease_up[:15]  + [f'GENE_{i:04d}' for i in rng.integers(700, 900, 10)]}
# Drug C: aggravating
drug_library['drug_C'] = {'up_genes': disease_up, 'down_genes': disease_down}
# Drug D: no overlap
drug_library['drug_D'] = {'up_genes': [f'GENE_{i:04d}' for i in range(800, 830)],
                          'down_genes': [f'GENE_{i:04d}' for i in range(900, 930)]}
len(drug_library)

## 3. Compute reversal scores

In [ ]:
from perturbation_layer.reversal_score import compute_reversal_score
scores = {}
for name, sig in drug_library.items():
    scores[name] = compute_reversal_score(disease_up, disease_down, sig['up_genes'], sig['down_genes'])
scores_df = pd.DataFrame({'compound': list(scores.keys()), 'reversal_score': list(scores.values())})
scores_df = scores_df.sort_values('reversal_score', ascending=False)
scores_df

## 4. Build reversal feature vectors (model-ready input)

In [ ]:
from perturbation_layer.reversal_features import build_reversal_feature_vector
fv = build_reversal_feature_vector(
    overall_reversal=0.8,
    cell_type_scores={'T_cell': 0.7, 'B_cell': 0.6, 'NK_cell': 0.9},
    pathway_scores={'IL6_signaling': 0.5, 'TNF_alpha': 0.7, 'NF_kB': 0.4},
)
print('Feature vector (6,):', fv)

## 5. Write artifact

In [ ]:
out = Path('artifacts/perturbations')
out.mkdir(parents=True, exist_ok=True)
scores_df.to_csv(out / 'reversal_scores.csv', index=False)
print(f'Wrote {out / "reversal_scores.csv"}')

**Next:** `06_candidate_ranking_and_validation.ipynb` fuses these reversal scores with KG and QML scores.